# Script 7: Discover Latent Topic Clusters, Assign Articles to Clusters, and Label Each Cluster

In this step, we embed the extracted `topic_annotation` feature and perform clustering using **K-Means**. To validate cluster quality, we use simple EDA heuristics, including a **silhouette score proxy** and **t-SNE visualization**.

### Top-Level Clusters

Our analysis reveals two primary types of article content:

1. **AI and Humanity** – Philosophical, ethical, and societal reflections on AI.  
2. **LLMs and AI Systems** – Technical, engineering, and research-focused discussions.

### Subclusters

We further subcluster each primary group using the same discovery process:

#### 1. AI and Humanity
- **1.1** Augmenting Humans with AI  
- **1.2** The Dangers of Over-Reliance on AI  

#### 2. LLMs and AI Systems
- **2.1** Machine Learning Fundamentals  
- **2.2** Building LLM Systems  
- **2.3** AI Consciousness 
- **2.5** Agentic AI


In [ ]:
import fenic as fc
from dotenv import load_dotenv

load_dotenv()

fc.configure_logging()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
            embedding_models={
                "large": fc.OpenAIModelConfig(
                    model_name="text-embedding-3-small",
                    rpm=3000,
                    tpm=1_000_000
                )
            }
        ),
    )

session = fc.Session.get_or_create(config)

##  Step 1: Embed `topic_annotation` with OpenAI's `text-embedding-small`. 
`extract_features.ipynb` uses text-embedding-large, but K-means can struggle with high-dimensionality data.

In [ ]:
source = (
    session.table("with_features")
           .select("url", "title", "text", "topic_annotation")
)

with_small_embedding = source.with_column(
    "small_topic_embedding",
    fc.semantic.embed(
        "topic_annotation",
    )
).cache()

with_small_embedding.write.save_as_table("with_small_embedding", mode="overwrite")

In [25]:
source = session.table("with_small_embedding")

## Step 2: Define Utility Methods to Explore Cluster Quality

For simplicity, we evaluate cluster coherence using a **silhouette-like metric** rather than the true silhouette coefficient. Our metric computes the difference between:
- A point's similarity to its own cluster centroid (cohesion)
- A point's similarity to the nearest other cluster centroid (separation)

**Formula**: `silhouette_score = distance_to_own_centroid - distance_to_nearest_other_centroid`

Where higher scores indicate better clustering (points are closer to their assigned centroid than to others).

This approach demonstrates Fenic's `EmbeddingType` and associated column functions for computing interpretable clustering metrics in a distributed setting.

In [26]:
def with_cluster_distances(
    clustered_df: fc.DataFrame,
    embedding_col: str,
    centroid_col: str,
    label_col: str
) -> fc.DataFrame:
    """
    Add both intra-cluster (distance to own centroid) and inter-cluster
    (distance to nearest other centroid) distances for each point.
    """
    # Step 1: Get all unique centroids with their labels
    clustered_df = clustered_df.with_column(
        centroid_col,
        fc.embedding.normalize(centroid_col)
    )
    centroids_df = (
        clustered_df
        .select(label_col, centroid_col)
        .drop_duplicates([label_col])
        .with_column_renamed(centroid_col, "other_centroid")
        .with_column_renamed(label_col, "other_label")
    )

    # Step 2: Cross join each point with ALL centroids
    df_with_all_centroids: fc.DataFrame = clustered_df.join(
        centroids_df,
        how="cross"
    )

    # Step 3: Compute distance to each centroid
    df_with_distances = df_with_all_centroids.with_column(
            "distance_to_centroid",
            fc.embedding.compute_similarity(
                embedding_col,
                "other_centroid",
                metric="dot"
            )
        ).filter(fc.col(label_col) != fc.col("other_label"))

    # Step 4: For each point, find distance to nearest OTHER centroid
    nearest_other_centroid = (
        df_with_distances
        .groupby("url")
        .agg(
            fc.max("distance_to_centroid").alias("distance_to_nearest_other_centroid")
        )
    )

    # Step 5: Add distance to own centroid
    df = (
        clustered_df
        .with_column(
            "distance_to_own_centroid",
            fc.embedding.compute_similarity(
                embedding_col,
                centroid_col,
                metric="dot"
            )
        )
    )

    # Step 6: Join back the inter-cluster distances
    final_df = df.join(
        nearest_other_centroid,
        on=["url"]
    )

    # Step 7: Compute separation score (silhouette-like)
    final_df = final_df.with_column(
        "separation_score",
        fc.col("distance_to_own_centroid") - fc.col("distance_to_nearest_other_centroid")
    )
    return final_df


In [27]:
def sweep_clusters(
    base_df: fc.DataFrame,
    embedding_col: str,
    label_col: str,
    centroid_col: str,
    k_values: list[int],
) -> list[dict]:
    """
    Test multiple cluster counts and display quality metrics.
    Returns list of results for further analysis.
    """
    results = []

    for k in k_values:
        print(f"\n{'='*50}")
        print(f"CLUSTERING RESULTS FOR k={k}")
        print(f"{'='*50}")

        # Cluster the data
        clustered = base_df.semantic.with_cluster_labels(
            by=embedding_col,
            num_clusters=k,
            label_column=label_col,
            centroid_column=centroid_col,
        )

        # Show per-cluster breakdown
        cluster_stats = _compute_cluster_stats(
            clustered, embedding_col, label_col, centroid_col
        )
        cluster_stats.select(
            label_col, "cluster_size", "mean_cohesion", "mean_separation", "mean_silhouette"
        ).show()

        # Compute overall quality
        overall_quality = _compute_overall_quality(
            clustered, embedding_col, label_col, centroid_col
        )

        print(f"\nOverall Metrics:")
        print(f"  Silhouette: {overall_quality['silhouette']:.3f}")
        print(f"  Cohesion:   {overall_quality['cohesion']:.3f}")
        print(f"  Separation: {overall_quality['separation']:.3f}")

        results.append({
            'k': k,
            'silhouette': overall_quality['silhouette'],
            'cohesion': overall_quality['cohesion'],
            'separation': overall_quality['separation']
        })

    # Show summary
    print(f"\n{'='*50}")
    print("SUMMARY")
    print(f"{'='*50}")
    best_result = max(results, key=lambda x: x['silhouette'])
    print(f"Best k: {best_result['k']} (silhouette: {best_result['silhouette']:.3f})")

    return results

def _compute_overall_quality(
    clustered_df: fc.DataFrame,
    embedding_col: str,
    label_col: str,
    centroid_col: str
) -> dict:
    """
    Compute only the metrics that matter for decision-making.
    """
    stats = _compute_cluster_stats(clustered_df, embedding_col, label_col, centroid_col)

    # Compute weighted averages (weighted by cluster size)
    result = (
        stats
        .select(
            fc.sum(fc.col("cluster_size") * fc.col("mean_cohesion")).alias("weighted_cohesion"),
            fc.sum(fc.col("cluster_size") * fc.col("mean_separation")).alias("weighted_separation"),
            fc.sum(fc.col("cluster_size") * fc.col("mean_silhouette")).alias("weighted_silhouette"),
            fc.sum("cluster_size").alias("total_size")
        )
        .with_column("overall_cohesion", fc.col("weighted_cohesion") / fc.col("total_size"))
        .with_column("overall_separation", fc.col("weighted_separation") / fc.col("total_size"))
        .with_column("overall_silhouette", fc.col("weighted_silhouette") / fc.col("total_size"))
    ).to_pylist()[0]

    return {
        'silhouette': result['overall_silhouette'],
        'cohesion': result['overall_cohesion'],
        'separation': result['overall_separation']
    }

def _compute_cluster_stats(
    clustered_df: fc.DataFrame,
    embedding_col: str,
    label_col: str,
    centroid_col: str
) -> fc.DataFrame:
    """
    Compute only essential per-cluster statistics.
    """
    df = with_cluster_distances(clustered_df, embedding_col, centroid_col, label_col)

    stats = (
        df
        .groupby(label_col)
        .agg(
            fc.mean("distance_to_own_centroid").alias("mean_cohesion"),
            fc.mean("distance_to_nearest_other_centroid").alias("mean_separation"),
            fc.mean("separation_score").alias("mean_silhouette"),
            fc.count("*").alias("cluster_size")
        )
    )

    return stats

In [29]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def visualize_embeddings(
    clustered_df: fc.DataFrame,
    embedding_col: str,
    label_col: str,
) -> None:
    """
    Visualize embeddings directly from a Fenic DataFrame.
    """
    df_pandas = clustered_df.to_pandas()

    embeddings = np.stack(df_pandas[embedding_col].values)
    labels = df_pandas[label_col].values

    _visualize_embeddings_2d(
        embeddings=embeddings,
        labels=labels,
        title=f"Article Embeddings Clustered (k={len(np.unique(labels))})"
    )

    plt.show()

def _visualize_embeddings_2d(
    embeddings: np.ndarray,
    labels: np.ndarray,
    title: str = "Embedding Visualization",
    figsize: tuple = (12, 8),
    random_state: int = 42
):
    """
    Visualize high-dimensional embeddings in 2D space.

    Args:
        embeddings: Array of shape (n_samples, n_features)
        labels: Optional cluster labels for coloring points
        title: Plot title
        figsize: Figure size
        random_state: Random seed for reproducibility

    Returns:
        2D coordinates and the plot figure
    """
    print(f"Visualizing {len(embeddings)} points using TSNE...")

    reducer = TSNE(
        n_components=2,
        random_state=random_state,
        perplexity=min(30, len(embeddings)//4),
        metric='cosine'
    )
    coords_2d = reducer.fit_transform(embeddings)


    fig, ax = plt.subplots(figsize=figsize)

    unique_labels = np.unique(labels)
    colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))

    for i, label in enumerate(unique_labels):
        mask = labels == label
        ax.scatter(
            coords_2d[mask, 0],
            coords_2d[mask, 1],
            c=[colors[i]],
            label=f'Cluster {label}',
            alpha=0.7,
            s=20
        )
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(f'TSNE Component 1')
    ax.set_ylabel(f'TSNE Component 2')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    return coords_2d, fig

## Step 3: Use the Utilities to Explore the Data

First, let's sweep across k-values to get some intuition on the "best" k value for our dataset. Our heuristics indicate that k=2 gives the best results.


In [ ]:
# sweep_clusters(
#     base_df=source,
#     embedding_col="small_topic_embedding",
#     label_col="topic_label",
#     centroid_col="topic_centroid",
#     k_values=[2, 3, 4, 5]
# )

Next, let's visualize the clusters with k=2 using t-SNE.

In [30]:
clustered_df = source.semantic.with_cluster_labels(
    by="small_topic_embedding",
    num_clusters=2,
    label_column="topic_label",
    centroid_column="topic_centroid"
).cache()

# visualize_embeddings(
#     clustered_df=source,
#     embedding_col="small_topic_embedding",
#     label_col="topic_label"
# )

Next, let's try sub-clustering the points.

In [31]:
cluster_0 = (
    clustered_df
    .filter(fc.col("topic_label") == 0)
)


cluster_1 = (
    clustered_df
    .filter(fc.col("topic_label") == 1)
)

In [ ]:
# sweep_clusters(
#     base_df=cluster_0,
#     embedding_col="small_topic_embedding",
#     label_col="sub_topic_label",
#     centroid_col="sub_topic_centroid",
#     k_values=[2, 3, 4, 5]
# )

Based on our heuristics, k=2 is the best for cluster 0.

In [32]:
clustered_0 = cluster_0.semantic.with_cluster_labels(
    by="small_topic_embedding",
    num_clusters=2,
    label_column="sub_topic_label",
    centroid_column="sub_topic_centroid"
)

# visualize_embeddings(
#     clustered_df=clustered_0,
#     embedding_col="small_topic_embedding",
#     label_col="sub_topic_label"
# )


In [ ]:
# sweep_clusters(
#     base_df=cluster_1,
#     embedding_col="small_topic_embedding",
#     label_col="sub_topic_label",
#     centroid_col="sub_topic_centroid",
#     k_values=[2, 3, 4, 5]
# )

Based on our heuristics, k=4 is best for cluster 1.

In [33]:
clustered_1 = cluster_1.semantic.with_cluster_labels(
    by="small_topic_embedding",
    num_clusters=4,
    label_column="sub_topic_label",
    centroid_column="sub_topic_centroid"
)

# visualize_embeddings(
#     clustered_df=clustered_1,
#     embedding_col="small_topic_embedding",
#     label_col="sub_topic_label"
# )

Based on our heuristics, we can see that k=2 is the best_value for cluster_1.
Our clusters certainly aren't very neat or perfect, but they can serve as weak signals. When combined with other signals we've extracted or user profiles, they can
help improve recommendation quality.

We can also omit labels for articles that are outliers/very far from centroids.

## Step 4: Define Prompts to Describe and Label Clusters

In [34]:
CLUSTER_DESCRIPTION_PROMPT = """
    You are analyzing a cluster of Medium article summaries, all related to artificial intelligence (AI). Your task is to synthesize a single, information-dense theme that captures what specifically unifies these articles beyond general AI relevance.

    Follow these steps to complete the task:

    1. Read all the article summaries carefully.
    2. Identify **specific** commonalities — recurring topics, concerns, techniques, or perspectives.
    3. Focus only on **subfields, applications, trends, or debates within AI** that appear across multiple summaries.
    4. Ignore superficial or vague mentions of "AI progress" or "AI in general." Your goal is to isolate a **precise**, **shared subject focus**.
    5. Synthesize a clear, **one- or two-sentence** theme that describes this shared focus.

    **Requirements for your output:**
    - Do *not* refer to individual summaries or mention article count.
    - Do *not* introduce or explain your answer — just state the theme.
    - Use exact language: no filler, no buzzwords, no generalizations.
    - The theme must reflect a **concrete** aspect of AI (e.g., regulation, LLM evaluation techniques, agent architectures), not just a broad area.

    Be concise, specific, and analytical.

    Article summaries: {summary}
"""

CLUSTER_NAME_PROMPT = """
    Your task is to generate a concise, specific tag that captures the core subtopic of AI discussed in the provided summary.

    Instructions:
    - The tag must clearly reflect the **main focus or trend** described in the summary.
    - Use **no more than 5 words**.
    - Avoid generic phrases (e.g., "AI trends") or vague buzzwords.
    - Prioritize **clarity, specificity, and memorability**.
    - Think of it like a hashtag or label that categorizes the content for someone scanning AI topics.

    Do not include any explanation — just return the tag and make sure its representative of the summary.

    Summary: {summary}
"""

In [35]:
def cluster_and_label(
    clustered_df: fc.DataFrame,
    embedding_col: str,
    label_col: str,
    centroid_col: str,
    cluster_name_col: str,
    cluster_description_col: str,
) -> fc.DataFrame:
    """
    Cluster and label a DataFrame using Fenic's clustering and labeling capabilities.
    """
    distance_from_centroid = (
        with_cluster_distances(
            clustered_df,
            embedding_col,
            centroid_col,
            label_col
        )
        .select(
            "url",
            "topic_annotation",
            label_col,
            "distance_to_own_centroid",
            embedding_col
        )
        .with_column_renamed("topic_annotation", "summary")
    )
    sql_query = """
        SELECT
            *,
            RANK() OVER (
                PARTITION BY {label_col}
                ORDER BY distance_to_own_centroid DESC
            ) AS cluster_rank
        FROM {{my_df}}
        """.format(label_col=label_col)

    ranked_distance_from_centroid = session.sql(
        sql_query,
        my_df=distance_from_centroid
    ).filter(
        fc.col("cluster_rank") <= 50
    ).with_column(
        embedding_col,
        fc.col(embedding_col).cast(
            fc.EmbeddingType(
                dimensions=1536,
                embedding_model="openai/text-embedding-3-small"
            )
        )
    )


    cluster_descriptions = (
        ranked_distance_from_centroid.group_by(label_col)
            .agg(
                fc.semantic.reduce(CLUSTER_DESCRIPTION_PROMPT).alias(cluster_description_col)
            )
        .with_column_renamed(
            cluster_description_col,
            "summary"
        )
        .with_column(cluster_name_col,
            fc.semantic.map(
                CLUSTER_NAME_PROMPT
            )
        )
        .cache()
    ).with_column_renamed(
        "summary",
        cluster_description_col,
    )
    cluster_descriptions.show()

    with_cluster_description = (
        clustered_df
        .join(cluster_descriptions, on=label_col)
    )

    return with_cluster_description

## Step 5: Label Clusters for Interpretability using `semantic.reduce()` and `semantic.map()`

`semantic.reduce()` hierarchically aggregates text within groups, similar to tabular aggregate functions like `count()` or `sum()`. 
We use `semantic.reduce()` to summarize the article descriptions within each cluster, then `semantic.map()` to generate concise labels for each cluster.

In [ ]:
labeled = cluster_and_label(
    clustered_df=clustered_df,
    embedding_col="small_topic_embedding",
    label_col="topic_label",
    centroid_col="topic_centroid",
    cluster_name_col="topic_name",
    cluster_description_col="topic_description"
).cache()

In [ ]:
clustered_0_labeled = cluster_and_label(
    clustered_df=clustered_0,
    embedding_col="small_topic_embedding",
    label_col="sub_topic_label",
    centroid_col="sub_topic_centroid",
    cluster_name_col="sub_topic_name",
    cluster_description_col="sub_topic_description"
).cache()

In [ ]:

clustered_1_labeled = cluster_and_label(
    clustered_df=clustered_1,
    embedding_col="small_topic_embedding",
    label_col="sub_topic_label",
    centroid_col="sub_topic_centroid",
    cluster_name_col="sub_topic_name",
    cluster_description_col="sub_topic_description"
).cache()

In [59]:
with_subtopic_labels = clustered_0_labeled.union(clustered_1_labeled)

## Step 6: Save Results

In [ ]:
with_topic_labels = (
    labeled
    .select(
        "url",
        "topic_name",
        "topic_description",
        "topic_centroid",
        "small_topic_embedding",
        "topic_annotation",
        "title",
        "text"
    )
)

with_subtopic_labels = (
    with_subtopic_labels
    .select(
        "url",
        "sub_topic_name",
        "sub_topic_description",
        "sub_topic_centroid",
        "topic_annotation",
        "title",
        "text"
    )
)

with_topic_labels.write.save_as_table("with_topic_labels", mode="overwrite")
with_subtopic_labels.write.save_as_table("with_subtopic_labels", mode="overwrite")


In [63]:
session.stop()